In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from cloud.functions.infrastructure.google.helper import load_google_credentials

credentials = load_google_credentials()

In [ ]:
from datetime import datetime, timezone, timedelta

from sebastian.shared.gmail.query_builder import GmailQueryBuilder

time_threshold = datetime.now(timezone.utc) - timedelta(hours=48)

query = (
    GmailQueryBuilder()
    .from_email("order-update@amazon.de")
    .subject("Ihr Paket kann bei DHL", exact=True)
    .after_date(time_threshold)
    .build()
)

In [ ]:
from pydantic import BaseModel


class MessageId(BaseModel):
    id: str
    thread_id: str

    @staticmethod
    def from_response(response: dict) -> "MessageId":
        return MessageId(id=response["id"], thread_id=response["threadId"])

In [8]:
import base64


def _extract_email_body(payload: dict) -> str:
    def _decode(inner: dict) -> str:
        return base64.urlsafe_b64decode(inner["body"]["data"].encode("utf-8")).decode(
            "utf-8"
        )

    if "parts" in payload:
        for part in payload["parts"]:
            if part["mimeType"] == "text/html":
                return _decode(part)
    else:
        return _decode(payload)

    return ""


class FullMailResponse(BaseModel):
    id: str
    threadId: str
    labelIds: list[str]
    snippet: str
    payload: str
    sizeEstimate: int
    historyId: str
    internalDate: str

    @staticmethod
    def from_response(response: dict) -> "FullMailResponse":
        return FullMailResponse(
            id=response["id"],
            threadId=response["threadId"],
            labelIds=response["labelIds"],
            snippet=response["snippet"],
            historyId=response["historyId"],
            internalDate=response["internalDate"],
            payload=_extract_email_body(response["payload"]),
            sizeEstimate=response["sizeEstimate"],
        )

In [9]:
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials


class GmailClient:
    def __init__(self, credentials: Credentials):
        self._service = build(
            "gmail", "v1", credentials=credentials, cache_discovery=False
        )

    def _fetch_message_ids(self, query: str) -> list[MessageId]:
        """Fetch message IDs matching the query"""
        response = self._service.users().messages().list(userId="me", q=query).execute()
        messages = response.get("messages", [])
        return [MessageId.from_response(msg) for msg in messages]

    def fetch_mails(self, query: str) -> list[FullMailResponse]:
        """Fetch full email messages matching the query"""
        message_ids = self._fetch_message_ids(query)
        emails = []
        for msg_id in message_ids:
            message = (
                self._service.users()
                .messages()
                .get(userId="me", id=msg_id.id, format="full")
                .execute()
            )
            emails.append(FullMailResponse.from_response(message))
        return emails

In [10]:
s = GmailClient(credentials)

In [11]:
ids = s._fetch_message_ids(query)
ids

[]

In [12]:
s.fetch_mails(query)

[]